# Multi BKT Example Using Simulated Grouped Data

This walkthrough mirrors the simple example (`02_simple_example.ipynb`) but uses grouped data and `MultiBKT`.

In this notebook, we will:

1. Simulate grouped BKT data with group- and KC-specific parameters.
2. Instantiate and fit a `MultiBKT` model.
3. Generate Numba-accelerated point-estimate predictions (`predict` and `predict_smoothed`).

This example stops after fitting and Numba-based predictions.


## Simulate Grouped BKT Data

We simulate students from multiple groups. Each group can have different BKT parameters, and in this example, parameters also vary across KCs.


In [21]:
from stanbkt.utils import sim_grouped_BKT

In [22]:
N_GROUPS = 3
N_KCS = 2

# rows=groups, cols=KCs
bkt_params = {
    "prior": [[0.20, 0.10], [0.35, 0.25], [0.50, 0.40]],
    "learn": [[0.03, 0.06], [0.05, 0.08], [0.08, 0.10]],
    "forget": [[0.01, 0.01], [0.02, 0.02], [0.02, 0.03]],
    "guess": [[0.25, 0.20], [0.20, 0.15], [0.15, 0.10]],
    "slip": [[0.10, 0.08], [0.08, 0.07], [0.06, 0.05]],
}

In [23]:
data_df = sim_grouped_BKT(
    n_students=60,
    n_problems=80,
    n_kcs=N_KCS,
    n_groups=N_GROUPS,
    frac=0.85,
    rng_seed=12345,
    **bkt_params,
)

In [24]:
data_df.head(10)

,student_id,problem_id,correct,timestamp,kc_id,group_id
0,stu_0,prob_0,0,2024-01-01 00:00:00,kc_1,group_0
1,stu_0,prob_1,1,2024-01-01 00:01:00,kc_0,group_0
2,stu_0,prob_2,0,2024-01-01 00:02:00,kc_1,group_0
3,stu_0,prob_3,1,2024-01-01 00:03:00,kc_0,group_0
4,stu_0,prob_4,1,2024-01-01 00:04:00,kc_0,group_0
5,stu_0,prob_6,0,2024-01-01 00:06:00,kc_1,group_0
6,stu_0,prob_7,1,2024-01-01 00:07:00,kc_1,group_0
7,stu_0,prob_8,1,2024-01-01 00:08:00,kc_1,group_0
8,stu_0,prob_9,1,2024-01-01 00:09:00,kc_0,group_0
9,stu_0,prob_10,0,2024-01-01 00:10:00,kc_1,group_0


In [25]:
data_df[["group_id", "kc_id"]].value_counts().sort_index()

group_id  kc_id
group_0   kc_0     658
          kc_1     700
group_1   kc_0     663
          kc_1     705
group_2   kc_0     656
          kc_1     698
Name: count, dtype: int64

`MultiBKT` expects long-format data including the standard columns (`student_id`, `problem_id`, `correct`, `timestamp`, `kc_id`) plus `group_id`.


In [26]:
required_cols = [
    "student_id",
    "problem_id",
    "correct",
    "timestamp",
    "kc_id",
    "group_id",
]
data_df[required_cols].head(5)

,student_id,problem_id,correct,timestamp,kc_id,group_id
0,stu_0,prob_0,0,2024-01-01 00:00:00,kc_1,group_0
1,stu_0,prob_1,1,2024-01-01 00:01:00,kc_0,group_0
2,stu_0,prob_2,0,2024-01-01 00:02:00,kc_1,group_0
3,stu_0,prob_3,1,2024-01-01 00:03:00,kc_0,group_0
4,stu_0,prob_4,1,2024-01-01 00:04:00,kc_0,group_0


## Instantiate and fit `MultiBKT`

As in the simple example, Stan code is compiled lazily on the first `fit(...)` call and cached for reuse.


In [27]:
from stanbkt.models import MultiBKT
from stanbkt.fits import FitMethod, MCMCFitOptions
from stanbkt.models import MultiPriors
from stanbkt.utils import VerbosityLevel

In [28]:
model = MultiBKT(
    fit_method=FitMethod.MCMC,
    verbose=VerbosityLevel.WARN,
)

fit_opts = MCMCFitOptions(
    seed=1234,
    iter_warmup=500,
    iter_sampling=500,
)

priors = MultiPriors(use_defaults=True)  # using default priors for all groups and KCs

Unlike the previous example (`02_simple_example.ipynb`), we will pass the entire DataFrame containing both KCs.


In [29]:
model.fit(data_df, stan_fit_options=fit_opts, priors=priors)

01:34:55 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

...

chain 2:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

01:35:08 - cmdstanpy - INFO - CmdStan done processing.


MultiBKT(fit_method=<FitMethod.MCMC: 'mcmc'>, verbose=<VerbosityLevel.WARN: 1>, is_fitted=True)

Generate a summary of the parameters' posterior distributions.


In [30]:
summary = model.summary()
summary

Mean      MCSE    StdDev       MAD  \
kc_id parameter                                                          
kc_1  lp__                   -921.644000  0.111816  3.014130  2.975060   
      logit_pi_know_group[1]   -3.688010  0.109315  2.540500  1.387180   
      logit_pi_know_group[2]   -0.950244  0.012550  0.625700  0.605511   
      logit_pi_know_group[3]    0.086050  0.010071  0.499394  0.490880   
      logit_learn_group[1]     -2.602730  0.005165  0.254314  0.260581   
...                                  ...       ...       ...       ...   
kc_0  guess[2]                  0.207365  0.000728  0.034745  0.033875   
      guess[3]                  0.148666  0.000753  0.038035  0.037235   
      slip[1]                   0.096801  0.000439  0.018771  0.018072   
      slip[2]                   0.049055  0.000273  0.012919  0.013067   
      slip[3]                   0.038828  0.000245  0.010606  0.010745   

                                    2.5%         50%       97.5%  ESS_bulk  \
kc_id parameter                                                              
kc_1  lp__                   -928.519000 -921.211000 -916.949000   735.702   
      logit_pi_know_group[1]  -11.232700   -2.898530   -1.095710   856.946   
      logit_pi_know_group[2]   -2.250440   -0.909651    0.173289  2618.950   
      logit_pi_know_group[3]   -0.896564    0.076857    1.089760  2490.820   
      logit_learn_group[1]     -3.117140   -2.596130   -2.128360  2482.720   
...                                  ...         ...         ...       ...   
kc_0  guess[2]                  0.143638    0.205421    0.282183  2324.310   
      guess[3]                  0.081344    0.146464    0.230444  2537.620   
      slip[1]                   0.063588    0.095322    0.137768  1956.280   
      slip[2]                   0.026140    0.048078    0.076607  2191.740   
      slip[3]                   0.020586    0.038407    0.060536  1763.590   

                              ESS_tail  ESS_bulk/s    R_hat  
kc_id parameter                                              
kc_1  lp__                    1195.080     53.7637  1.00892  
      logit_pi_know_group[1]   754.213     62.6239  1.00260  
      logit_pi_know_group[2]  1451.420    191.3880  1.00151  
      logit_pi_know_group[3]  1261.560    182.0240  1.00033  
      logit_learn_group[1]    1381.690    181.4320  1.00168  
...                                ...         ...      ...  
kc_0  guess[2]                1465.820    267.0700  1.00503  
      guess[3]                1514.460    291.5800  1.00272  
      slip[1]                 1404.730    224.7820  0.99963  
      slip[2]                 1187.940    251.8370  1.00133  
      slip[3]                 1060.960    202.6410  1.00153  

[62 rows x 11 columns]

In [31]:
summary.loc["kc_0"]  # summary for kc_0

,Mean,MCSE,StdDev,MAD,2.5%,50%,97.5%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
parameter,,,,,,,,,,,
lp__,-837.483000,0.103154,2.923860,2.806810,-844.394000,-837.181000,-832.782000,761.343,1273.140,87.4806,1.00898
logit_pi_know_group[1],-1.190040,0.020063,0.703330,0.602880,-2.693300,-1.136770,-0.039843,1826.940,1026.930,209.9210,1.00702
logit_pi_know_group[2],0.587203,0.010804,0.547383,0.546459,-0.421654,0.577471,1.704440,2615.630,1472.040,300.5430,1.01056
logit_pi_know_group[3],1.047820,0.012930,0.616644,0.610961,-0.104917,1.014700,2.311310,2378.180,1538.900,273.2600,1.00230
logit_learn_group[1],-3.297660,0.008497,0.378123,0.364899,-4.101890,-3.270960,-2.613490,2102.400,1302.740,241.5720,1.00235
logit_learn_group[2],-2.789640,0.008074,0.359271,0.359296,-3.569940,-2.763580,-2.151780,2164.080,1123.390,248.6590,1.00545
logit_learn_group[3],-2.132480,0.007017,0.316328,0.308663,-2.798890,-2.120160,-1.547510,2081.410,1468.860,239.1600,1.00242
logit_forget_group[1],-5.596310,0.060199,1.513660,1.133490,-9.828310,-5.241170,-3.705400,1129.330,594.778,129.7630,1.00668
logit_forget_group[2],-3.428370,0.006929,0.321330,0.324496,-4.088770,-3.416490,-2.826330,2312.840,1552.710,265.7520,1.00384


## Numba-based point-estimate predictions

`predict(...)` and `predict_smoothed(...)` use Numba-compiled routines internally for fast inference with Bayesian point estimates for the parameters.


In [32]:
from stanbkt.utils.data_utils import ColumnNames

col_mapping = {
    ColumnNames.STUDENT_ID: "student_id",
    ColumnNames.PROBLEM_ID: "problem_id",
    ColumnNames.CORRECTNESS: "correct",
    ColumnNames.ORDER: "timestamp",
    ColumnNames.KC_ID: "kc_id",
    ColumnNames.GROUP: "group_id",
}

In [33]:
predictions = model.predict(
    data_df,
    column_mapping=col_mapping,
    point_estimate="mean",
    parallel=True,
    fast_math=True,
)

predictions.head(20)

,kc_id,student_id,problem_id,pKnow,pCorrectness,correct
0,kc_1,stu_0,prob_0,0.071660,0.216030,0
1,kc_1,stu_0,prob_2,0.079721,0.221914,0
2,kc_1,stu_0,prob_6,0.080807,0.222707,0
3,kc_1,stu_0,prob_7,0.080955,0.222814,1
4,kc_1,stu_0,prob_8,0.370694,0.434298,1
5,kc_1,stu_0,prob_10,0.775405,0.729702,0
6,kc_1,stu_0,prob_12,0.352623,0.421109,0
7,kc_1,stu_0,prob_17,0.130594,0.259047,0
8,kc_1,stu_0,prob_19,0.088058,0.227999,0
9,kc_1,stu_0,prob_20,0.081948,0.223539,0


In [34]:
smoothed_predictions = model.predict_smoothed(
    data_df,
    column_mapping=col_mapping,
    point_estimate="mean",
    parallel=True,
    fast_math=True,
)

smoothed_predictions.head(20)

,kc_id,student_id,problem_id,pKnow,pCorrectness,correct
0,kc_1,stu_0,prob_0,0.000113,0.163807,0
1,kc_1,stu_0,prob_2,0.000460,0.164060,0
2,kc_1,stu_0,prob_6,0.002953,0.165879,0
3,kc_1,stu_0,prob_7,0.021264,0.179245,1
4,kc_1,stu_0,prob_8,0.021265,0.179246,1
5,kc_1,stu_0,prob_10,0.002959,0.165884,0
6,kc_1,stu_0,prob_12,0.000468,0.164066,0
7,kc_1,stu_0,prob_17,0.000129,0.163819,0
8,kc_1,stu_0,prob_19,0.000083,0.163785,0
9,kc_1,stu_0,prob_20,0.000077,0.163780,0
